In [1]:
# Cell 1 - Install ChromaDB
import sys
!{sys.executable} -m pip install chromadb -q

In [12]:
# Cell 2 - Load data files, deduplicate, and align embeddings by paper_id
import json
import numpy as np

with open("data/processed/qbio_papers.json", "r") as f:
    raw_papers = json.load(f)

embeddings_raw = np.load("data/processed/qbio_embeddings.npy")

with open("data/processed/embedding_info.json", "r") as f:
    emb_info = json.load(f)

# Build a lookup: paper_id -> embedding row index
id_to_index = {pid: i for i, pid in enumerate(emb_info["paper_ids"])}

# Deduplicate papers and pull the correct embedding for each
seen = set()
papers = []
embeddings_list = []

for p in raw_papers:
    pid = p["paper_id"]
    if pid not in seen and pid in id_to_index:
        seen.add(pid)
        papers.append(p)
        embeddings_list.append(embeddings_raw[id_to_index[pid]])

embeddings = np.array(embeddings_list)

print(f"Raw papers:         {len(raw_papers)}")
print(f"After dedup:        {len(papers)}")
print(f"Duplicates removed: {len(raw_papers) - len(papers)}")
print(f"Embeddings shape:   {embeddings.shape}")

Raw papers:         55301
After dedup:        55300
Duplicates removed: 1
Embeddings shape:   (55300, 384)


In [14]:
# Cell 3 - Verify each paper has a matching embedding by paper_id
mismatches = 0

for i, p in enumerate(papers):
    expected_id = emb_info["paper_ids"][id_to_index[p["paper_id"]]]
    if expected_id != p["paper_id"]:
        mismatches += 1

if mismatches == 0:
    print("All paper IDs aligned - safe to proceed.")
else:
    print(f"{mismatches} mismatches found - do NOT proceed.")

All paper IDs aligned - safe to proceed.


In [15]:
# Cell 4 - Initialize ChromaDB
import chromadb

client = chromadb.PersistentClient(path="data/chroma")

# Remove existing collection if rerunning this notebook
try:
    client.delete_collection("qbio_papers")
    print("Existing collection deleted.")
except:
    pass

collection = client.create_collection(
    name="qbio_papers",
    metadata={"hnsw:space": "cosine"}
)

print(f"Collection 'qbio_papers' created.")

Existing collection deleted.
Collection 'qbio_papers' created.


In [16]:
# Cell 5 - Insert records into ChromaDB in batches of 500
from tqdm import tqdm

BATCH_SIZE = 500

for start in tqdm(range(0, len(papers), BATCH_SIZE), desc="Inserting"):
    end = min(start + BATCH_SIZE, len(papers))
    
    batch_papers = papers[start:end]
    batch_embeddings = embeddings[start:end]

    collection.add(
        ids        = [p["paper_id"] for p in batch_papers],
        embeddings = batch_embeddings.tolist(),
        metadatas  = [
            {
                "title"      : p.get("title", ""),
                "year"       : str(p.get("year", "")),
                "categories" : p.get("categories", "")
            }
            for p in batch_papers
        ]
    )

print(f"Inserted {collection.count()} records into ChromaDB.")

Inserting: 100%|██████████| 111/111 [00:24<00:00,  4.59it/s]

Inserted 55300 records into ChromaDB.


In [ ]:
# # Reload raw papers without dedup for inspection
# import json

# with open("data/processed/qbio_papers.json", "r") as f:
#     raw_papers = json.load(f)

# from collections import Counter

# id_counts = Counter(p["paper_id"] for p in raw_papers)
# duplicates = {pid: count for pid, count in id_counts.items() if count > 1}

# print(f"Total papers (raw):  {len(raw_papers)}")
# print(f"Unique paper IDs:    {len(id_counts)}")
# print(f"Duplicate IDs found: {len(duplicates)}")
# print(f"\nTop 10 most duplicated:")
# for pid, count in sorted(duplicates.items(), key=lambda x: -x[1])[:10]:
#     print(f"  {pid}: appears {count} times")

Total papers (raw):  55301
Unique paper IDs:    55300
Duplicate IDs found: 1

Top 10 most duplicated:
  math-ph/0608014: appears 2 times


In [ ]:
# # Find the exact positions of the duplicate
# for i, p in enumerate(papers):
#     if p["paper_id"] == "math-ph/0608014":
#         print(f"Index {i}: {p['paper_id']}")

Index 52373: math-ph/0608014
Index 52490: math-ph/0608014


In [20]:
# Inspect the duplicate paper
dup_id = "math-ph/0608014"

matches = [p for p in raw_papers if p["paper_id"] == dup_id]

print(f"Number of occurrences: {len(matches)}")
print()

for i, p in enumerate(matches):
    print(f"--- Occurrence {i+1} ---")
    for key, value in p.items():
        print(f"  {key}: {value}")
    print()

Number of occurrences: 2

--- Occurrence 1 ---
  paper_id: math-ph/0608014
  title: Gauss-Vanicek Spectral Analysis of the Sepkoski Compendium: No New Life Cycles
  abstract: New periods can emerge from data as a byproduct of incorrect processing or even the method applied. In one such recent instance, a new life cycle with a 62+-3 Myr period was reportedly found (about trend) in genus variations from the Sepkoski compendium, the world most complete fossil record. The approach that led to reporting this period was based on Fourier method of spectral analysis. I show here that no such period is found when the original data set is considered rigorously and processed in the Gauss-Vanicek spectral analysis. I also demonstrate that data altering can boost spectral power up to a nearly 100 percent increase in the signal range, thus introducing artificial, "99 percent significant" periods as seen in the corresponding variance-spectra of noise. Besides geology and paleontology, virtually all s

In [17]:
# Cell 6 - Validate the ChromaDB index
from sentence_transformers import SentenceTransformer

count = collection.count()
print(f"Total records in ChromaDB: {count}")

# Load the same model used to generate the embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# Run a test query
test_query = "How do neural oscillations contribute to memory consolidation?"
query_vector = model.encode(test_query).tolist()

results = collection.query(query_embeddings=[query_vector], n_results=3)

print(f"\nTest query: '{test_query}'")
print("Top 3 results:")
for i, (pid, meta) in enumerate(zip(results["ids"][0], results["metadatas"][0])):
    print(f"  {i+1}. [{pid}] {meta['title'][:80]}")

Total records in ChromaDB: 55300


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\rajlu\anaconda3\envs\fairsearch\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rajlu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Test query: 'How do neural oscillations contribute to memory consolidation?'
Top 3 results:
  1. [1703.01357] A Computational Model of Systems Memory Consolidation and Reconsolidation
  2. [2402.01605] A Lyapunov theory demonstrating a fundamental limit on the speed of systems cons
  3. [2404.02938] A Coupled Neural Field Model for the Standard Consolidation Theory


In [18]:
# Cell 7 - Confirm ChromaDB files saved to disk
import os

chroma_path = "data/chroma"
total_size = 0

for f in os.listdir(chroma_path):
    fpath = os.path.join(chroma_path, f)
    size = os.path.getsize(fpath)
    total_size += size
    print(f"  {f}: {size / 1024 / 1024:.1f} MB")

print(f"\nTotal size: {total_size / 1024 / 1024:.1f} MB")

  .gitkeep: 0.0 MB
  02f4f02a-ea64-42f6-bd2c-08f72989db36: 0.0 MB
  7ec08b57-26fc-47a2-bf63-611e3c4274af: 0.0 MB
  a81d74d2-8af9-4a05-81f5-e45e16ffef99: 0.0 MB
  bbe3414b-0a46-4d04-a4d1-e620d1cd41f6: 0.0 MB
  chroma.sqlite3: 31.7 MB
  d961c434-7f42-440b-82e9-2ad0da870928: 0.0 MB

Total size: 31.7 MB


In [19]:
# Cell 7 - Confirm ChromaDB files saved to disk
import os

chroma_path = "data/chroma"
total_size = 0

for root, dirs, files in os.walk(chroma_path):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        total_size += size
        print(f"  {os.path.relpath(fpath, chroma_path)}: {size / 1024 / 1024:.1f} MB")

print(f"\nTotal size: {total_size / 1024 / 1024:.1f} MB")

  .gitkeep: 0.0 MB
  chroma.sqlite3: 31.7 MB
  02f4f02a-ea64-42f6-bd2c-08f72989db36\data_level0.bin: 83.1 MB
  02f4f02a-ea64-42f6-bd2c-08f72989db36\header.bin: 0.0 MB
  02f4f02a-ea64-42f6-bd2c-08f72989db36\index_metadata.pickle: 2.0 MB
  02f4f02a-ea64-42f6-bd2c-08f72989db36\length.bin: 0.2 MB
  02f4f02a-ea64-42f6-bd2c-08f72989db36\link_lists.bin: 0.4 MB
  7ec08b57-26fc-47a2-bf63-611e3c4274af\data_level0.bin: 87.9 MB
  7ec08b57-26fc-47a2-bf63-611e3c4274af\header.bin: 0.0 MB
  7ec08b57-26fc-47a2-bf63-611e3c4274af\index_metadata.pickle: 2.1 MB
  7ec08b57-26fc-47a2-bf63-611e3c4274af\length.bin: 0.2 MB
  7ec08b57-26fc-47a2-bf63-611e3c4274af\link_lists.bin: 0.4 MB
  a81d74d2-8af9-4a05-81f5-e45e16ffef99\data_level0.bin: 83.1 MB
  a81d74d2-8af9-4a05-81f5-e45e16ffef99\header.bin: 0.0 MB
  a81d74d2-8af9-4a05-81f5-e45e16ffef99\index_metadata.pickle: 2.0 MB
  a81d74d2-8af9-4a05-81f5-e45e16ffef99\length.bin: 0.2 MB
  a81d74d2-8af9-4a05-81f5-e45e16ffef99\link_lists.bin: 0.4 MB
  bbe3414b-0a46-4d04-a